# I05 - Transfer Learning and Fine-tuning

**Author:** Karthik Arjun  
**LinkedIn:** [karthik-arjun-a5b4a258](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Created:** March 2026  
**Level:** Intermediate

---

## Learning Objectives

By the end of this lesson, you will:
1. Understand transfer learning concepts
2. Use pre-trained models from ImageNet
3. Implement feature extraction
4. Master fine-tuning strategies
5. Apply domain adaptation techniques

---

## Prerequisites

- Completed I04 (Advanced CNN Architectures)
- Understanding of CNNs and training
- Familiarity with different architectures

---

In [ ]:
# Import required libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import (
    ResNet50, VGG16, MobileNetV2, EfficientNetB0
)
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print(f"TensorFlow version: {tf.__version__}")

## 1. What is Transfer Learning?

### The Core Idea

**Transfer Learning:** Use knowledge learned from one task to improve performance on a related task

**Why Transfer Learning?**
- Limited training data
- Expensive to train from scratch
- Pre-trained models learn general features
- Faster convergence
- Better performance

### Two Main Approaches

1. **Feature Extraction**
   - Freeze pre-trained layers
   - Use as fixed feature extractor
   - Train only new classifier

2. **Fine-tuning**
   - Unfreeze some layers
   - Continue training with small learning rate
   - Adapt features to new task

In [ ]:
# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normalize to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Resize images to 96x96 (closer to ImageNet size)
x_train_resized = tf.image.resize(x_train, [96, 96]).numpy()
x_test_resized = tf.image.resize(x_test, [96, 96]).numpy()

# Use subset for faster training
x_train_small = x_train_resized[:5000]
y_train_small = y_train[:5000]

print(f"Training samples: {x_train_small.shape}")
print(f"Test samples: {x_test_resized.shape}")

# Class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

## 2. Feature Extraction with Pre-trained Models

### Using Pre-trained Models

**Popular pre-trained models:**
- ResNet50, ResNet101
- VGG16, VGG19
- MobileNetV2 (efficient)
- EfficientNet (state-of-the-art)

**All trained on ImageNet:**
- 1.2 million images
- 1000 classes
- General visual features

### Feature Extraction Strategy

1. Load pre-trained model (without top layer)
2. Freeze all layers
3. Add custom classifier
4. Train only new layers

In [ ]:
# Load pre-trained ResNet50 (without top classification layer)
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(96, 96, 3)
)

# Freeze all layers in base model
base_model.trainable = False

print(f"Base model has {len(base_model.layers)} layers")
print(f"Trainable: {base_model.trainable}")
print(f"\nBase model output shape: {base_model.output_shape}")

In [ ]:
# Build feature extraction model
def create_feature_extraction_model(base_model, num_classes=10):
    """Create model using pre-trained base as feature extractor"""
    
    inputs = keras.Input(shape=(96, 96, 3))
    
    # Pre-trained base (frozen)
    x = base_model(inputs, training=False)
    
    # Global pooling
    x = layers.GlobalAveragePooling2D()(x)
    
    # Custom classifier
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs, name='FeatureExtraction')
    return model

# Create model
feature_model = create_feature_extraction_model(base_model)

# Count trainable parameters
trainable_params = sum([tf.size(w).numpy() for w in feature_model.trainable_weights])
total_params = feature_model.count_params()

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

In [ ]:
# Train feature extraction model
print("Training with feature extraction...\n")

feature_model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_feature = feature_model.fit(
    x_train_small, y_train_small,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Evaluate
test_loss, test_acc = feature_model.evaluate(x_test_resized, y_test, verbose=0)
print(f"\nFeature Extraction Test Accuracy: {test_acc:.4f}")

## 3. Fine-tuning Pre-trained Models

### Fine-tuning Strategy

**Steps:**
1. Start with feature extraction (train classifier)
2. Unfreeze top layers of base model
3. Continue training with very small learning rate
4. Monitor for overfitting

**Best Practices:**
- Use small learning rate (1e-5 to 1e-4)
- Unfreeze gradually (top layers first)
- More data = more layers to unfreeze
- Monitor validation metrics closely

### Which Layers to Unfreeze?

**General rule:**
- Early layers: Generic features (edges, textures)
- Later layers: Task-specific features
- Unfreeze later layers first

In [ ]:
# Fine-tuning: Unfreeze top layers
print("Setting up fine-tuning...\n")

# Unfreeze the base model
base_model.trainable = True

# Freeze all layers except the last 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Count trainable layers
trainable_layers = sum([layer.trainable for layer in base_model.layers])
print(f"Total layers in base model: {len(base_model.layers)}")
print(f"Trainable layers: {trainable_layers}")
print(f"Frozen layers: {len(base_model.layers) - trainable_layers}")

In [ ]:
# Recompile with lower learning rate
feature_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),  # Much smaller learning rate
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nFine-tuning the model...\n")

# Continue training (fine-tuning)
history_finetune = feature_model.fit(
    x_train_small, y_train_small,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Evaluate after fine-tuning
test_loss, test_acc = feature_model.evaluate(x_test_resized, y_test, verbose=0)
print(f"\nFine-tuned Test Accuracy: {test_acc:.4f}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Combine histories
feature_epochs = len(history_feature.history['loss'])
finetune_epochs = len(history_finetune.history['loss'])
total_epochs = feature_epochs + finetune_epochs

# Loss
axes[0].plot(range(1, feature_epochs + 1), 
            history_feature.history['loss'], 
            label='Feature Extraction (train)', linewidth=2)
axes[0].plot(range(1, feature_epochs + 1), 
            history_feature.history['val_loss'], 
            label='Feature Extraction (val)', linewidth=2, linestyle='--')
axes[0].plot(range(feature_epochs + 1, total_epochs + 1), 
            history_finetune.history['loss'], 
            label='Fine-tuning (train)', linewidth=2)
axes[0].plot(range(feature_epochs + 1, total_epochs + 1), 
            history_finetune.history['val_loss'], 
            label='Fine-tuning (val)', linewidth=2, linestyle='--')
axes[0].axvline(x=feature_epochs, color='red', linestyle=':', linewidth=2, label='Start Fine-tuning')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss: Feature Extraction → Fine-tuning', 
                 fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(range(1, feature_epochs + 1), 
            history_feature.history['accuracy'], 
            label='Feature Extraction (train)', linewidth=2)
axes[1].plot(range(1, feature_epochs + 1), 
            history_feature.history['val_accuracy'], 
            label='Feature Extraction (val)', linewidth=2, linestyle='--')
axes[1].plot(range(feature_epochs + 1, total_epochs + 1), 
            history_finetune.history['accuracy'], 
            label='Fine-tuning (train)', linewidth=2)
axes[1].plot(range(feature_epochs + 1, total_epochs + 1), 
            history_finetune.history['val_accuracy'], 
            label='Fine-tuning (val)', linewidth=2, linestyle='--')
axes[1].axvline(x=feature_epochs, color='red', linestyle=':', linewidth=2, label='Start Fine-tuning')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training Accuracy: Feature Extraction → Fine-tuning', 
                 fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Comparing Different Pre-trained Models

### Model Selection Guide

| Model | Parameters | Speed | Accuracy | Best For |
|-------|-----------|-------|----------|----------|
| **MobileNetV2** | 3.5M | Fast | Good | Mobile, edge devices |
| **ResNet50** | 25M | Medium | Very Good | General purpose |
| **VGG16** | 138M | Slow | Good | Feature extraction |
| **EfficientNetB0** | 5.3M | Fast | Excellent | Best efficiency |
| **ResNet101** | 44M | Slow | Excellent | High accuracy needed |

In [ ]:
# Compare different pre-trained models
def create_transfer_model(base_model_class, input_shape=(96, 96, 3), num_classes=10):
    """Create transfer learning model with any base"""
    base = base_model_class(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base.trainable = False
    
    inputs = keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    return model

# Create models with different bases
models_comparison = {
    'MobileNetV2': MobileNetV2,
    'ResNet50': ResNet50,
    'EfficientNetB0': EfficientNetB0
}

print("Model Comparison:")
print("=" * 70)
for name, base_class in models_comparison.items():
    model = create_transfer_model(base_class)
    params = model.count_params()
    print(f"{name:20} | Parameters: {params:,}")
print("=" * 70)

## 5. Data Augmentation for Transfer Learning

### Why Augmentation Matters More

With transfer learning:
- Often have limited data
- Pre-trained on different domain
- Augmentation helps adaptation

**Recommended augmentations:**
- Random flips
- Random rotations (small angles)
- Random zoom
- Color jittering
- Random crops

In [ ]:
# Create data augmentation pipeline
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.1)
])

# Model with augmentation
def create_augmented_transfer_model(base_model, num_classes=10):
    inputs = keras.Input(shape=(96, 96, 3))
    
    # Data augmentation (only during training)
    x = data_augmentation(inputs)
    
    # Pre-trained base
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    return model

print("Model with data augmentation created")

## 6. Practical Tips for Transfer Learning

### Decision Flow

**Small dataset, similar to ImageNet:**
- Use feature extraction only
- Don't fine-tune

**Small dataset, different from ImageNet:**
- Use feature extraction
- Fine-tune top layers carefully

**Large dataset, similar to ImageNet:**
- Start with feature extraction
- Fine-tune many layers

**Large dataset, different from ImageNet:**
- Fine-tune entire network
- Or train from scratch

### Common Mistakes to Avoid

1. **High learning rate during fine-tuning** → Destroys pre-trained weights
2. **Unfreezing too early** → Unstable training
3. **Wrong input preprocessing** → Poor performance
4. **Not using data augmentation** → Overfitting
5. **Forgetting to set training=False** → Batch norm issues

## Summary

In this lesson, you learned:

1. ✅ Transfer learning concepts and benefits
2. ✅ Feature extraction with frozen layers
3. ✅ Fine-tuning strategies and best practices
4. ✅ Comparing different pre-trained models
5. ✅ Data augmentation for transfer learning

**Key Takeaways:**
- Transfer learning saves time and improves performance
- Start with feature extraction, then fine-tune
- Use very small learning rates for fine-tuning
- Choose model based on speed/accuracy trade-off
- Data augmentation is crucial with limited data

**Next Steps:**
- Move to I06 - Object Detection and Segmentation
- Try transfer learning on your own datasets
- Experiment with different pre-trained models

---

**References:**
- Yosinski et al. (2014): "How transferable are features in deep neural networks?"
- Kornblith et al. (2019): "Do Better ImageNet Models Transfer Better?"
- TensorFlow Transfer Learning Guide
- Keras Applications Documentation

---

## Learning Progress Tracker

Use this section to track your learning progress for this lesson.

### Completion Status
- [ ] Lesson completed
- [ ] All code cells executed successfully
- [ ] Understood all key concepts
- [ ] Completed practice exercises (if any)

### Dates
- **First Completed:** ____/____/____
- **Last Reviewed:** ____/____/____
- **Next Review:** ____/____/____ (Recommended: 1 week, 1 month, 3 months)

### Understanding Level
Rate your understanding (1-5): _____ / 5

- 1 = Need to review completely
- 2 = Understood basics, need more practice
- 3 = Good understanding, minor gaps
- 4 = Strong understanding, can explain to others
- 5 = Mastered, can apply in projects

### Notes & Reflections
```
Write your notes here:
- What concepts were challenging?
- What was interesting or surprising?
- How can you apply this in projects?
- Questions to explore further?




```

### Key Concepts to Remember (I05)
- [ ] Pre-trained models
- [ ] Fine-tuning strategies
- [ ] Domain adaptation
- [ ] Few-shot learning

---